In [1]:
pip install pyspark

In [2]:
pip install pandas

In [4]:
# Import the necessary libraries for both PySpark and Pandas
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import upper, sum, count
import pandas as pd

In [6]:
# Create a PySpark SparkSession
spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

In [8]:
# --- Create a simple DataFrame (df) for our example ---
# PySpark data definition
data = [
    Row(name='Alice', dept='Engineering', salary=75000),
    Row(name='Bob', dept='HR', salary=60000),
    Row(name='Charlie', dept='Engineering', salary=80000),
    Row(name='David', dept='Sales', salary=90000),
    Row(name='Eve', dept='HR', salary=65000),
    Row(name='Frank', dept='Sales', salary=95000),
]
# Create the PySpark DataFrame
df = spark.createDataFrame(data)



In [9]:
# Now, let's perform some common data manipulation operations.

# Display the first 20 rows of the DataFrame
# PySpark:
df.show()

# Pandas:
# print(pdf.head())

+-------+-----------+------+
|   name|       dept|salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|         HR| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 90000|
|    Eve|         HR| 65000|
|  Frank|      Sales| 95000|
+-------+-----------+------+



In [10]:
# Get the column names
# PySpark:
print("PySpark Columns:", df.columns)
# Pandas:
# print("Pandas Columns:", pdf.columns)

PySpark Columns: ['name', 'dept', 'salary']


In [11]:
# Print the schema of the DataFrame to see its structure
# PySpark:
df.printSchema()
# Pandas:
# print(pdf.info())

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)



In [12]:
# Show summary statistics for the DataFrame's numerical columns
# PySpark:
df.describe().show()
# Pandas:
# print(pdf.describe())

+-------+-----+-----------+------------------+
|summary| name|       dept|            salary|
+-------+-----+-----------+------------------+
|  count|    6|          6|                 6|
|   mean| NULL|       NULL|           77500.0|
| stddev| NULL|       NULL|13693.063937629151|
|    min|Alice|Engineering|             60000|
|    max|Frank|      Sales|             95000|
+-------+-----+-----------+------------------+



In [13]:
# Select specific columns
# PySpark:
df.select("name", "dept").show()
# Pandas:
# print(pdf[['name', 'dept']])

+-------+-----------+
|   name|       dept|
+-------+-----------+
|  Alice|Engineering|
|    Bob|         HR|
|Charlie|Engineering|
|  David|      Sales|
|    Eve|         HR|
|  Frank|      Sales|
+-------+-----------+



In [14]:
# Filter rows based on a single condition (salary over 70000)
# PySpark:
df.filter(df['salary'] > 70000).show()
# Pandas:
# print(pdf[pdf['salary'] > 70000])

+-------+-----------+------+
|   name|       dept|salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|Charlie|Engineering| 80000|
|  David|      Sales| 90000|
|  Frank|      Sales| 95000|
+-------+-----------+------+



In [15]:
# Filter with multiple conditions
# PySpark:
df.filter((df['dept'] == 'Engineering') & (df['salary'] > 70000)).show()
# Pandas:
# print(pdf[(pdf['dept'] == 'Engineering') & (pdf['salary'] > 70000)])

+-------+-----------+------+
|   name|       dept|salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|Charlie|Engineering| 80000|
+-------+-----------+------+



In [16]:
# --- The withColumn command ---
# Use withColumn to add a new column or update an existing one.
# Here, we add a new 'bonus' column calculated from the 'salary' column.
# PySpark:
df.withColumn("bonus", df['salary'] * 0.1).show()
# Pandas:
# pdf['bonus'] = pdf['salary'] * 0.1
# print(pdf)

+-------+-----------+------+------+
|   name|       dept|salary| bonus|
+-------+-----------+------+------+
|  Alice|Engineering| 75000|7500.0|
|    Bob|         HR| 60000|6000.0|
|Charlie|Engineering| 80000|8000.0|
|  David|      Sales| 90000|9000.0|
|    Eve|         HR| 65000|6500.0|
|  Frank|      Sales| 95000|9500.0|
+-------+-----------+------+------+



In [17]:
# Another example: updating an existing column.
# Here we convert the 'dept' column to all uppercase.
# PySpark:
df.withColumn('dept', upper(df['dept'])).show()
# Pandas:
# pdf['dept'] = pdf['dept'].str.upper()
# print(pdf)

+-------+-----------+------+
|   name|       dept|salary|
+-------+-----------+------+
|  Alice|ENGINEERING| 75000|
|    Bob|         HR| 60000|
|Charlie|ENGINEERING| 80000|
|  David|      SALES| 90000|
|    Eve|         HR| 65000|
|  Frank|      SALES| 95000|
+-------+-----------+------+



In [18]:
# Group by a column and calculate the sum of salaries for each group
# PySpark:
df.groupBy('dept').sum('salary').show()
# Pandas:
# print(pdf.groupby('dept')['salary'].sum())

+-----------+-----------+
|       dept|sum(salary)|
+-----------+-----------+
|Engineering|     155000|
|         HR|     125000|
|      Sales|     185000|
+-----------+-----------+



In [19]:
# Group and apply multiple aggregation functions
# PySpark:
df.groupBy('dept').agg(
    sum('salary').alias('total_salary'),
    count('name').alias('employee_count')
).show()
# Pandas:
# print(pdf.groupby('dept').agg(total_salary=('salary', 'sum'), employee_count=('name', 'count')))

+-----------+------------+--------------+
|       dept|total_salary|employee_count|
+-----------+------------+--------------+
|Engineering|      155000|             2|
|         HR|      125000|             2|
|      Sales|      185000|             2|
+-----------+------------+--------------+



In [20]:
# Join with a second, smaller DataFrame (df2)
df2 = spark.createDataFrame([
    Row(name='Alice', project='Project A'),
    Row(name='Bob', project='Project B'),
])


# Join the two DataFrames on the 'name' column
# PySpark:
df.join(df2, on='name', how='inner').show()
# Pandas:
# print(pd.merge(pdf, pdf2, on='name', how='inner'))

+-----+-----------+------+---------+
| name|       dept|salary|  project|
+-----+-----------+------+---------+
|Alice|Engineering| 75000|Project A|
|  Bob|         HR| 60000|Project B|
+-----+-----------+------+---------+



In [ ]:
# Create a temporary view to run SQL queries directly on the data (PySpark-only feature)
df.createOrReplaceTempView("emp")

# Run a SQL query on the temporary view
# PySpark:
query_result = spark.sql("SELECT dept, sum(salary) as total_salary FROM emp GROUP BY dept")
query_result.show()


# Pandas (Not applicable, but you could use a library like pandasql if you needed this functionality)
# from pandasql import sqldf
# query_result_pandas = sqldf("SELECT dept, sum(salary) as total_salary FROM pdf GROUP BY dept", locals())
# print(query_result_pandas)

# Stop the SparkSession when you're done to release resources
# spark.stop()
